In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

!pip install torch lifelines -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 11.3 MB/s eta 0:00:00


In [3]:
import os
import json
import copy
import warnings
import itertools

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from lifelines.utils import concordance_index

In [4]:
PROCESSED_DATA_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/"
UNICOX_OUTPUT_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/UnivariateCox/"
RESULTS_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/UnivariateCox/"

os.makedirs(UNICOX_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_PATH = UNICOX_OUTPUT_DIR + "hcc_UniCox_train_selected.csv"

TUNING_RESULTS_PATH = RESULTS_DIR + "hcc_UniCox_deepsurv_tuning_results.csv"
TUNING_FOLD_DETAILS_PATH = RESULTS_DIR + "hcc_UniCox_deepsurv_tuning_fold_details.csv"
BEST_PARAMS_JSON_PATH = RESULTS_DIR + "hcc_UniCox_deepsurv_best_hyperparameters.json"
BEST_PARAMS_CSV_PATH = RESULTS_DIR + "hcc_UniCox_deepsurv_best_hyperparameters.csv"

DURATION_COL = "OS_MONTHS"
EVENT_COL = "OS_EVENT"

In [5]:
train_data = pd.read_csv(TRAIN_PATH)

print("Train data shape:", train_data.shape)
train_data.head()

Train data shape: (1085, 52)


,METASTASIS_Yes,CURATIVE_INTENT_SURGERY_Yes,CURATIVE_INTENT_SURGERY_No,METASTASIS_No,MAJOR_LIVER_RESECTION_,RESECTION_MARGIN_,RESECTION_MARGIN_R0,C_N_Yes,PNI_,LVI_,...,TUMOR_STAGE_Poorly differentiated,TUMOR_STAGE_Well differentiated,MUT_SHQ1,CNA_XPO1,CNA_REL,MUT_KRAS,NEOADJUVANT_SYSTEMIC_THERAPY_No,MUT_FGFR4,OS_MONTHS,OS_EVENT
0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,82.296,0
1,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,51.324,1
2,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,6.600,1
3,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,24.480,0
4,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,14.592,1


In [6]:
print("Train missing values:", int(train_data.isna().sum().sum()))

print("\nTrain OS_EVENT distribution:")
print(train_data[EVENT_COL].value_counts())

Train missing values: 0

Train OS_EVENT distribution:
OS_EVENT
1    641
0    444
Name: count, dtype: int64


In [7]:
for id_col in ["PATIENT_ID", "SAMPLE_ID"]:
    assert id_col not in train_data.columns, f"{id_col} found in train_data"

print("ID leakage guard passed.")

ID leakage guard passed.


In [8]:
COLS_TO_DROP = []

if COLS_TO_DROP:
    train_data = train_data.drop(columns=COLS_TO_DROP)

train_data.shape

(1085, 52)

In [9]:
DEPENDENT_COLUMNS = ["OS_MONTHS", "OS_EVENT"]
ID_COLUMNS = ["PATIENT_ID", "SAMPLE_ID"]

GENES = [
    col for col in train_data.columns
    if col not in DEPENDENT_COLUMNS + ID_COLUMNS
]

print("Feature count:", len(GENES))
GENES[:10]

Feature count: 50


['METASTASIS_Yes',
 'CURATIVE_INTENT_SURGERY_Yes',
 'CURATIVE_INTENT_SURGERY_No',
 'METASTASIS_No',
 'MAJOR_LIVER_RESECTION_',
 'RESECTION_MARGIN_',
 'RESECTION_MARGIN_R0',
 'C_N_Yes',
 'PNI_',
 'LVI_']

In [10]:
X = train_data[GENES].copy()
y = train_data[DEPENDENT_COLUMNS].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1085, 50)
y shape: (1085, 2)


In [11]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_SPLITS = 5
RANDOM_STATE = 42

print("Device:", DEVICE)
print("N_SPLITS:", N_SPLITS)

Device: cpu
N_SPLITS: 5


In [12]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

In [13]:
class DeepSurv(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate, activation_name):
        super(DeepSurv, self).__init__()

        activation_map = {
            "ReLU": nn.ReLU,
            "LeakyReLU": nn.LeakyReLU,
            "ELU": nn.ELU,
            "SELU": nn.SELU
        }

        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(activation_map[activation_name]())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            current_dim = hidden_dim

        self.feature_extractor = nn.Sequential(*layers)
        self.output_layer = nn.Linear(current_dim, 1)

    def forward(self, x):
        x = self.feature_extractor(x)
        return self.output_layer(x)

In [14]:
def nll_loss(preds, durations, events):
    order = torch.argsort(durations, descending=True)

    hazards = preds[order].reshape(-1)
    events = events[order].reshape(-1)

    log_risk = torch.logcumsumexp(hazards, dim=0)
    uncensored_loss = hazards - log_risk
    censored_loss = uncensored_loss * events

    return -torch.sum(censored_loss) / torch.sum(events)

In [15]:
def build_optimizer(model, optimizer_name, learning_rate, weight_decay):
    if optimizer_name == "Adam":
        return optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "AdamW":
        return optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "RMSprop":
        return optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

In [16]:
def evaluate_c_index(model, X_eval, y_eval):
    model.eval()

    X_eval_tensor = torch.tensor(X_eval.values, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        preds = model(X_eval_tensor).cpu().numpy().reshape(-1)

    c_index = concordance_index(
        y_eval[DURATION_COL].values,
        -preds,
        y_eval[EVENT_COL].values
    )

    return float(c_index)

In [17]:
def train_one_fold(X_train_fold, y_train_fold, X_valid_fold, y_valid_fold, config):
    set_seed(RANDOM_STATE)

    X_train_tensor = torch.tensor(X_train_fold.values, dtype=torch.float32)
    y_train_duration = torch.tensor(y_train_fold[DURATION_COL].values, dtype=torch.float32)
    y_train_event = torch.tensor(y_train_fold[EVENT_COL].values, dtype=torch.float32)

    train_dataset = TensorDataset(X_train_tensor, y_train_duration, y_train_event)
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True
    )

    model = DeepSurv(
        input_dim=X_train_fold.shape[1],
        hidden_layers=config["hidden_layers"],
        dropout_rate=config["dropout_rate"],
        activation_name=config["activation"]
    ).to(DEVICE)

    optimizer = build_optimizer(
        model=model,
        optimizer_name=config["optimizer"],
        learning_rate=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )

    best_val_c_index = -np.inf
    best_epoch = 0
    best_state_dict = None
    epochs_without_improvement = 0

    for epoch in range(config["epochs"]):
        model.train()

        for batch_X, batch_duration, batch_event in train_loader:
            batch_X = batch_X.to(DEVICE)
            batch_duration = batch_duration.to(DEVICE)
            batch_event = batch_event.to(DEVICE)

            optimizer.zero_grad()
            preds = model(batch_X)
            loss = nll_loss(preds, batch_duration, batch_event)
            loss.backward()
            optimizer.step()

        val_c_index = evaluate_c_index(model, X_valid_fold, y_valid_fold)

        if val_c_index > best_val_c_index:
            best_val_c_index = val_c_index
            best_epoch = epoch + 1
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= config["patience"]:
            break

    return best_val_c_index, best_epoch, best_state_dict

In [18]:
HIDDEN_LAYER_OPTIONS = [
    [64],
    [128],
    [128, 64],
    [256, 128]
]

DROPOUT_OPTIONS = [0.0, 0.2]
BATCH_SIZE_OPTIONS = [64, 128]
ACTIVATION_OPTIONS = ["ReLU", "LeakyReLU"]
OPTIMIZER_OPTIONS = ["Adam", "AdamW"]
WEIGHT_DECAY_OPTIONS = [0.0, 1e-4]
LEARNING_RATE_OPTIONS = [1e-4, 1e-3, 5e-3]
PATIENCE_OPTIONS = [10, 20]
EPOCH_OPTIONS = [75, 100]

In [19]:
all_configurations = list(itertools.product(
    HIDDEN_LAYER_OPTIONS,
    DROPOUT_OPTIONS,
    BATCH_SIZE_OPTIONS,
    ACTIVATION_OPTIONS,
    OPTIMIZER_OPTIONS,
    WEIGHT_DECAY_OPTIONS,
    LEARNING_RATE_OPTIONS,
    PATIENCE_OPTIONS,
    EPOCH_OPTIONS
))

print("Total grid combinations:", len(all_configurations))

Total grid combinations: 1536


In [20]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

trial_results = []
fold_details = []

In [21]:
for trial_counter, config_tuple in enumerate(all_configurations, start=1):
    (
        hidden_layers,
        dropout_rate,
        batch_size,
        activation,
        optimizer_name,
        weight_decay,
        learning_rate,
        patience,
        epochs
    ) = config_tuple

    config = {
        "hidden_layers": hidden_layers,
        "dropout_rate": dropout_rate,
        "batch_size": batch_size,
        "activation": activation,
        "optimizer": optimizer_name,
        "weight_decay": weight_decay,
        "learning_rate": learning_rate,
        "patience": patience,
        "epochs": epochs
    }

    print(f"\nRunning trial {trial_counter}/{len(all_configurations)}")
    print(config)

    fold_scores = []
    fold_best_epochs = []

    for fold_idx, (train_index, valid_index) in enumerate(
        skf.split(X, y[EVENT_COL].astype(int).values), start=1
    ):
        X_train_fold = X.iloc[train_index].copy()
        X_valid_fold = X.iloc[valid_index].copy()

        y_train_fold = y.iloc[train_index].copy()
        y_valid_fold = y.iloc[valid_index].copy()

        try:
            best_val_c_index, best_epoch, _ = train_one_fold(
                X_train_fold,
                y_train_fold,
                X_valid_fold,
                y_valid_fold,
                config
            )

            fold_scores.append(float(best_val_c_index))
            fold_best_epochs.append(int(best_epoch))

            fold_details.append({
                "trial": trial_counter,
                "fold": fold_idx,
                "status": "success",
                "hidden_layers": json.dumps(hidden_layers),
                "dropout_rate": dropout_rate,
                "batch_size": batch_size,
                "activation": activation,
                "optimizer": optimizer_name,
                "weight_decay": weight_decay,
                "learning_rate": learning_rate,
                "patience": patience,
                "epochs": epochs,
                "best_epoch_in_fold": best_epoch,
                "c_index": float(best_val_c_index),
                "error_message": ""
            })

        except Exception as e:
            fold_details.append({
                "trial": trial_counter,
                "fold": fold_idx,
                "status": "failed",
                "hidden_layers": json.dumps(hidden_layers),
                "dropout_rate": dropout_rate,
                "batch_size": batch_size,
                "activation": activation,
                "optimizer": optimizer_name,
                "weight_decay": weight_decay,
                "learning_rate": learning_rate,
                "patience": patience,
                "epochs": epochs,
                "best_epoch_in_fold": np.nan,
                "c_index": np.nan,
                "error_message": str(e)
            })

    successful_folds = len(fold_scores)
    failed_folds = N_SPLITS - successful_folds

    trial_results.append({
        "trial": trial_counter,
        "hidden_layers": json.dumps(hidden_layers),
        "dropout_rate": dropout_rate,
        "batch_size": batch_size,
        "activation": activation,
        "optimizer": optimizer_name,
        "weight_decay": weight_decay,
        "learning_rate": learning_rate,
        "patience": patience,
        "epochs": epochs,
        "successful_folds": successful_folds,
        "failed_folds": failed_folds,
        "mean_best_epoch": np.mean(fold_best_epochs) if successful_folds > 0 else np.nan,
        "mean_c_index": np.mean(fold_scores) if successful_folds > 0 else np.nan,
        "std_c_index": np.std(fold_scores) if successful_folds > 0 else np.nan,
        "status": "ok" if successful_folds > 0 else "all_failed"
    })

    if successful_folds > 0:
        print("Fold C-indexes:", [round(score, 4) for score in fold_scores])
        print("Mean C-index:", round(np.mean(fold_scores), 4))
    else:
        print("All folds failed for this trial.")

Streaming output truncated to the last 5000 lines.

Running trial 537/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'Adam', 'weight_decay': 0.0, 'learning_rate': 0.005, 'patience': 10, 'epochs': 75}
Fold C-indexes: [0.7534, 0.7262, 0.7431, 0.7575, 0.7254]
Mean C-index: 0.7411

Running trial 538/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'Adam', 'weight_decay': 0.0, 'learning_rate': 0.005, 'patience': 10, 'epochs': 100}
Fold C-indexes: [0.7534, 0.7262, 0.7431, 0.7575, 0.7254]
Mean C-index: 0.7411

Running trial 539/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_size': 128, 'activation': 'LeakyReLU', 'optimizer': 'Adam', 'weight_decay': 0.0, 'learning_rate': 0.005, 'patience': 20, 'epochs': 75}
Fold C-indexes: [0.7534, 0.7262, 0.7431, 0.7575, 0.7254]
Mean C-index: 0.7411

Running trial 540/1536
{'hidden_layers': [128], 'dropout_rate': 0.0, 'batch_s

In [22]:
results_df = pd.DataFrame(trial_results).sort_values(
    ["mean_c_index", "successful_folds"],
    ascending=[False, False]
).reset_index(drop=True)

fold_details_df = pd.DataFrame(fold_details)

results_df.head(10)

,trial,hidden_layers,dropout_rate,batch_size,activation,optimizer,weight_decay,learning_rate,patience,epochs,successful_folds,failed_folds,mean_best_epoch,mean_c_index,std_c_index,status
0,1079,"[128, 64]",0.2,128,ReLU,Adam,0.0001,0.005,20,75,5,0,32.4,0.746464,0.010380,ok
1,1080,"[128, 64]",0.2,128,ReLU,Adam,0.0001,0.005,20,100,5,0,32.4,0.746464,0.010380,ok
2,1355,"[256, 128]",0.2,64,ReLU,Adam,0.0000,0.005,20,75,5,0,32.2,0.746164,0.011641,ok
3,1356,"[256, 128]",0.2,64,ReLU,Adam,0.0000,0.005,20,100,5,0,32.2,0.746164,0.011641,ok
4,1379,"[256, 128]",0.2,64,ReLU,AdamW,0.0000,0.005,20,75,5,0,32.2,0.746164,0.011641,ok
5,1380,"[256, 128]",0.2,64,ReLU,AdamW,0.0000,0.005,20,100,5,0,32.2,0.746164,0.011641,ok
6,1536,"[256, 128]",0.2,128,LeakyReLU,AdamW,0.0001,0.005,20,100,5,0,34.6,0.746139,0.015507,ok
7,443,[128],0.0,64,LeakyReLU,Adam,0.0000,0.005,20,75,5,0,33.0,0.746133,0.010261,ok
8,444,[128],0.0,64,LeakyReLU,Adam,0.0000,0.005,20,100,5,0,33.0,0.746133,0.010261,ok
9,467,[128],0.0,64,LeakyReLU,AdamW,0.0000,0.005,20,75,5,0,33.0,0.746133,0.010261,ok


In [23]:
results_df.to_csv(TUNING_RESULTS_PATH, index=False)
fold_details_df.to_csv(TUNING_FOLD_DETAILS_PATH, index=False)

TUNING_RESULTS_PATH, TUNING_FOLD_DETAILS_PATH

('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/UnivariateCox/hcc_UniCox_deepsurv_tuning_results.csv',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/UnivariateCox/hcc_UniCox_deepsurv_tuning_fold_details.csv')

In [24]:
valid_results = results_df.dropna(subset=["mean_c_index"]).copy()

if valid_results.empty:
    raise RuntimeError("All DeepSurv CV configurations failed. Check TUNING_FOLD_DETAILS_PATH for errors.")

best_result = valid_results.iloc[0].copy()
best_hidden_layers = json.loads(best_result["hidden_layers"])

BEST_HYPERPARAMETERS = {
    "hidden_layers": best_hidden_layers,
    "dropout_rate": float(best_result["dropout_rate"]),
    "batch_size": int(best_result["batch_size"]),
    "activation": str(best_result["activation"]),
    "optimizer": str(best_result["optimizer"]),
    "weight_decay": float(best_result["weight_decay"]),
    "learning_rate": float(best_result["learning_rate"]),
    "patience": int(best_result["patience"]),
    "epochs": int(best_result["epochs"]),
    "mean_best_epoch": float(best_result["mean_best_epoch"]),
    "mean_c_index": float(best_result["mean_c_index"])
}

print("Best hyperparameters:")
print(BEST_HYPERPARAMETERS)

Best hyperparameters:
{'hidden_layers': [128, 64], 'dropout_rate': 0.2, 'batch_size': 128, 'activation': 'ReLU', 'optimizer': 'Adam', 'weight_decay': 0.0001, 'learning_rate': 0.005, 'patience': 20, 'epochs': 75, 'mean_best_epoch': 32.4, 'mean_c_index': 0.7464639634046063}


In [25]:
with open(BEST_PARAMS_JSON_PATH, "w") as file:
    json.dump(BEST_HYPERPARAMETERS, file, indent=2)

best_params_df = pd.DataFrame([{
    "hidden_layers": json.dumps(BEST_HYPERPARAMETERS["hidden_layers"]),
    "dropout_rate": BEST_HYPERPARAMETERS["dropout_rate"],
    "batch_size": BEST_HYPERPARAMETERS["batch_size"],
    "activation": BEST_HYPERPARAMETERS["activation"],
    "optimizer": BEST_HYPERPARAMETERS["optimizer"],
    "weight_decay": BEST_HYPERPARAMETERS["weight_decay"],
    "learning_rate": BEST_HYPERPARAMETERS["learning_rate"],
    "patience": BEST_HYPERPARAMETERS["patience"],
    "epochs": BEST_HYPERPARAMETERS["epochs"],
    "mean_best_epoch": BEST_HYPERPARAMETERS["mean_best_epoch"],
    "mean_c_index": BEST_HYPERPARAMETERS["mean_c_index"]
}])

best_params_df.to_csv(BEST_PARAMS_CSV_PATH, index=False)

BEST_PARAMS_JSON_PATH, BEST_PARAMS_CSV_PATH

('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/UnivariateCox/hcc_UniCox_deepsurv_best_hyperparameters.json',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/UnivariateCox/hcc_UniCox_deepsurv_best_hyperparameters.csv')

In [26]:
print("Best hidden layers:", BEST_HYPERPARAMETERS["hidden_layers"])
print("Best dropout rate:", BEST_HYPERPARAMETERS["dropout_rate"])
print("Best batch size:", BEST_HYPERPARAMETERS["batch_size"])
print("Best activation:", BEST_HYPERPARAMETERS["activation"])
print("Best optimizer:", BEST_HYPERPARAMETERS["optimizer"])
print("Best weight decay:", BEST_HYPERPARAMETERS["weight_decay"])
print("Best learning rate:", BEST_HYPERPARAMETERS["learning_rate"])
print("Best patience:", BEST_HYPERPARAMETERS["patience"])
print("Best epochs:", BEST_HYPERPARAMETERS["epochs"])
print("Best mean validation C-index:", round(BEST_HYPERPARAMETERS["mean_c_index"], 4))

Best hidden layers: [128, 64]
Best dropout rate: 0.2
Best batch size: 128
Best activation: ReLU
Best optimizer: Adam
Best weight decay: 0.0001
Best learning rate: 0.005
Best patience: 20
Best epochs: 75
Best mean validation C-index: 0.7465


**UnivariateCox DeepSurv Training Summary**

The updated UnivariateCox DeepSurv hyperparameter tuning notebook has completed successfully. This run used the revised UnivariateCox-selected feature set, where features were selected primarily using survival-associated statistical significance and then prioritized by C-index strength. The DeepSurv model was then optimized using grid-search cross-validation on the training data only.

This is an important result because the updated UnivariateCox DeepSurv model achieved a stronger validation C-index than the earlier UnivariateCox DeepSurv run.

**Training Setup**

The DeepSurv model was trained using the UnivariateCox selected feature matrix. The model objective was the Cox partial likelihood loss, meaning the network learned patient-level relative risk scores while accounting for censoring.

The notebook performed full grid-search tuning across neural-network and training hyperparameters.

| Hyperparameter Category | Search Space |
|---|---|
| Hidden layers | `[64]`, `[128]`, `[128, 64]`, `[256, 128]` |
| Dropout | `0.0`, `0.2` |
| Batch size | `64`, `128` |
| Activation | `ReLU`, `LeakyReLU` |
| Optimizer | `Adam`, `AdamW` |
| Weight decay | `0.0`, `0.0001` |
| Learning rate | `0.0001`, `0.001`, `0.005` |
| Patience | `10`, `20` |
| Epochs | `75`, `100` |

The model selection criterion was mean validation C-index across 5 cross-validation folds.

**Best Hyperparameter Configuration**

| Hyperparameter | Best Value |
|---|---|
| Hidden layers | `[128, 64]` |
| Dropout rate | `0.2` |
| Batch size | `128` |
| Activation | `ReLU` |
| Optimizer | `Adam` |
| Weight decay | `0.0001` |
| Learning rate | `0.005` |
| Early stopping patience | `20` |
| Epochs | `75` |
| Mean best epoch | `32.4` |
| Mean validation C-index | `0.7465` |
| Standard deviation C-index | `0.0104` |
| Successful folds | `5/5` |

The best model used a two-layer neural architecture with 128 neurons in the first hidden layer and 64 neurons in the second hidden layer. This suggests that the updated UnivariateCox feature set benefited from a moderately expressive nonlinear model rather than a very shallow single-layer network.

**Key Takeaways From the Top Trials**

The top two rows were essentially the same configuration except for `epochs = 75` versus `epochs = 100`.

| Trial | Hidden Layers | Dropout | Batch | Activation | Optimizer | LR | Epochs | Mean C-index |
|---|---|---:|---:|---|---|---:|---:|---:|
| 1079 | `[128, 64]` | `0.2` | `128` | ReLU | Adam | `0.005` | `75` | `0.7465` |
| 1080 | `[128, 64]` | `0.2` | `128` | ReLU | Adam | `0.005` | `100` | `0.7465` |

Both achieved the same mean validation C-index, and both had the same mean best epoch of `32.4`. This tells us that increasing the maximum epoch count from 75 to 100 did not improve validation performance because early stopping found the best model much earlier.

So the selected `75` epochs is reasonable and efficient.

**Early Stopping Behavior**

The mean best epoch was `32.4`, even though the maximum epoch setting was `75`. This indicates that the model generally reached its best validation performance around epoch 32 and did not need the full training duration.

This is a good sign because:

| Observation | Meaning |
|---|---|
| Best epoch around 32 | Model learns useful survival signal relatively early |
| Patience 20 selected | Training benefits from waiting before stopping |
| 75 and 100 epochs tied | Longer training does not add value |
| Dropout selected | Some regularization improves validation stability |

The use of dropout `0.2` and weight decay `0.0001` suggests that regularization helped the DeepSurv model generalize better on this feature set.

**Architecture Interpretation**

The best architecture was `[128, 64]`.

This is meaningful because:

| Architecture Choice | Interpretation |
|---|---|
| Two hidden layers | Nonlinear interactions among features are useful |
| 128 then 64 neurons | Model benefits from moderate compression |
| ReLU activation | Standard nonlinear activation worked better than LeakyReLU in top trial |
| Dropout 0.2 | Regularization was useful |
| Batch size 128 | Stable training with larger mini-batches |
| Adam optimizer | Adam slightly outperformed AdamW for the best run |

The top trials also included `[256, 128]` models, but those did not exceed the best `[128, 64]` configuration. This suggests that increasing model size beyond `[128, 64]` did not provide additional benefit and may risk unnecessary complexity.

**Top-Trial Pattern**

Several top-performing models shared common traits:

| Common Trait | Observation |
|---|---|
| Learning rate | Top trials consistently used `0.005` |
| Patience | Top trials consistently used `20` |
| Dropout | Many top trials used `0.2` |
| Architecture | `[128, 64]` and `[256, 128]` dominated |
| Activation | ReLU appeared frequently among top models |
| Epochs | 75 and 100 often tied due to early stopping |

This shows the performance was not random. The search repeatedly favored faster learning rate, longer patience, and moderately/deeply layered architectures.

**Validation Performance**

The best mean validation C-index was:

`0.7465`

This is strong for the training-only validation stage. It means that across the five validation folds, the model correctly ranked survival risk for approximately 74.6% of comparable patient pairs.

The standard deviation was:

`0.0104`

This is relatively low, indicating stable fold-to-fold performance. The model did not perform well in only one fold while failing in others; instead, the validation result appears consistent across the training data.

**Comparison With Updated UnivariateCox CoxPH Training**

The updated UnivariateCox CoxPH model had:

| Model | Validation C-index |
|---|---:|
| CoxPH | `0.7377` |
| DeepSurv | `0.7465` |

At the validation stage, DeepSurv outperformed CoxPH by approximately `0.0088`.

This suggests that, after the revised UnivariateCox feature selection, the nonlinear DeepSurv model may be capturing additional survival-relevant structure that CoxPH does not fully capture.

However, final judgment still depends on held-out test evaluation. The model must now be saved using the best hyperparameters and evaluated on the test set.

**Comparison With Earlier UnivariateCox DeepSurv Run**

The earlier UnivariateCox DeepSurv tuning result was approximately:

| Run | Best Validation C-index |
|---|---:|
| Previous UnivariateCox DeepSurv | `0.7398` |
| Updated UnivariateCox DeepSurv | `0.7465` |

The updated feature-selection strategy improved DeepSurv validation performance by about `0.0067`.

This is encouraging because although the updated CoxPH test performance dropped, the revised feature set appears more favorable for nonlinear DeepSurv learning.

**Important Outcome**

The updated UnivariateCox DeepSurv model is currently the strongest training-validation model observed in the revised UnivariateCox rerun.

Best result:

| Item | Value |
|---|---|
| Trial | `1079` |
| Mean validation C-index | `0.7465` |
| Fold success | `5/5` |
| Mean best epoch | `32.4` |
| Std C-index | `0.0104` |
| Architecture | `[128, 64]` |
| Regularization | Dropout `0.2`, weight decay `0.0001` |

**What Happens Next**

The next step is to run the saving-best-model notebook using these selected hyperparameters:

| Parameter | Use This |
|---|---|
| Hidden layers | `[128, 64]` |
| Dropout rate | `0.2` |
| Batch size | `128` |
| Activation | `ReLU` |
| Optimizer | `Adam` |
| Weight decay | `0.0001` |
| Learning rate | `0.005` |
| Patience | `20` |
| Epochs | `75` |

After saving the final model, run the DeepSurv evaluation notebook to compute:

| Evaluation Metric | Needed |
|---|---|
| Test C-index | Yes |
| IBS | Yes |
| AUC at 1 year | Yes |
| AUC at 3 years | Yes |
| AUC at 5 years | Yes |
| Mean time-dependent AUC | Yes |
| Kaplan-Meier curves | Yes |
| Log-rank p-value | Yes |

**Conclusion**

The updated UnivariateCox DeepSurv tuning run is successful and promising. The model selected a regularized two-layer architecture with `[128, 64]`, dropout `0.2`, Adam optimizer, learning rate `0.005`, and early stopping patience `20`. It achieved a mean validation C-index of `0.7465`, outperforming the updated UnivariateCox CoxPH validation result.

The key conclusion is that the revised statistically grounded UnivariateCox feature set appears to work better with DeepSurv than with CoxPH at the validation stage. This supports the idea that DeepSurv may be capturing nonlinear interactions among the updated clinical and genomic predictors. Final confirmation will come from the held-out test evaluation.